# Lecture : Graph-based Visualization

## Lab 01 : (Standard) Linear PCA

### Xavier Bresson   


In [ ]:
# For Google Colaboratory
import sys, os
if 'google.colab' in sys.modules:
    # mount google drive
    from google.colab import drive
    drive.mount('/content/gdrive')
    path_to_file = '/content/gdrive/My Drive/CS5284_2026_codes/04_Visualization'
    print(path_to_file)
    # change current path to the folder containing "path_to_file"
    os.chdir(path_to_file)
    !pwd
    

In [ ]:
# Load libraries
import numpy as np
import scipy.io
from matplotlib import pyplot
import matplotlib.pyplot as plt
import time
import sys; sys.path.insert(0, 'lib/')
import scipy.sparse.linalg
# import scipy.ndimage
from lib.utils import compute_pca
import warnings; warnings.filterwarnings("ignore")


In [ ]:
# Data matrix 
mat = scipy.io.loadmat('datasets/ellipsoide_standardpca.mat')
X = mat['X']
n = X.shape[0]
d = X.shape[1]
print('n,d:',n,d)


# PCA for simple artificial dataset

## Question 1: PCA as EVD of covariance matrix
Complete the code below to perform PCA on the dataset X
- step 1: compute the covariance matrix of zero-centered data
- step 2: compute the largest 5 eigenvectors of the covariance matrix. Hint: [scipy.sparse.linalg.eigsh](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.eigsh.html)
- 

In [ ]:
# Compute largest 5 eigenvectors/eigenvalues
nb_pca = 5
lamb, U = None, None
########## Your code here #########
# compute eigenvalues lamb, and eigenvectors U

###################################

EVec = U[:,::-1] # largest = index 0
EVal = lamb[::-1]

# Principal Components
Xpc = None
########## Your code here #########
# compute Xpc, the principal components of X

###################################


# Principal Directions
v1 = EVec[:2,0]
v2 = EVec[:2,1]
print(v1,v2,EVal[:2])


In [ ]:
# Visualize the dataset
plt.figure(1)
size_vertex_plot = 10
plt.scatter(X[:,0], X[:,1], s=size_vertex_plot*np.ones(n))
plt.title('Data dimentionality=' + str(d))
plt.axis('equal')
plt.show()

plt.figure(2)
size_vertex_plot = 10
plt.scatter(Xzc[:,0], Xzc[:,1], s=size_vertex_plot*np.ones(n))
ct = 3
k=0; p=ct * EVal[k]/ 10000.* v1
plt.quiver(0.0, 0.0, p[0], p[1], scale=1., units='xy', color='r') 
k=1; p=ct * EVal[k]/ 1000.* v2
plt.quiver(0.0, 0.0, p[0], p[1], scale=1., units='xy', color='g') 
plt.title('Principal Directions')
plt.axis('equal')
plt.show()

plt.figure(3)
size_vertex_plot = 10
plt.scatter(Xpc[:,0], Xpc[:,1], s=size_vertex_plot*np.ones(n))
plt.title('Principal Components = \nData projected on the Principal Directions')
plt.axis('equal')
plt.show()


# PCA for real-world faces

In [ ]:
# Yale Faces
mat = scipy.io.loadmat('datasets/yale_faces.mat')
X = mat['X']
n = X.shape[0]
d = X.shape[1]
Nx = mat['Nx_downscale'].squeeze()
Ny = mat['Ny_downscale'].squeeze()
Cgt = C = mat['Cgt'].squeeze()
print(n,d,Nx,Ny)

plt.figure(1)
rotated_img = scipy.ndimage.rotate(np.reshape(X[0,:],[Nx,Ny]), -90)
plt.subplot(131).imshow(rotated_img, interpolation='nearest', cmap='Greys_r')
plt.axis('equal')
plt.axis('off')
rotated_img = scipy.ndimage.rotate(np.reshape(X[10,:],[Nx,Ny]), -90)
plt.subplot(132).imshow(rotated_img, interpolation='nearest', cmap='Greys_r')
plt.axis('equal')
plt.axis('off')
rotated_img = scipy.ndimage.rotate(np.reshape(X[20,:],[Nx,Ny]), -90)
plt.subplot(133).imshow(rotated_img, interpolation='nearest', cmap='Greys_r')
plt.axis('equal')
plt.axis('off')
plt.show()


In [ ]:
# Run Standard PCA
nb_pca = 3
nb_pca = 10
[PC,PD,EnPD] = compute_pca(X,nb_pca)
Xvis = PC[:,0]
Yvis = PC[:,1]
Zvis = PC[:,2]

# Plot
plt.figure(11)
plt.plot(EnPD)
plt.title('Variances of Principal Directions')

# 2D Plot
plt.figure(2)
size_vertex_plot = 50
plt.scatter(Xvis, Yvis, s=size_vertex_plot*np.ones(n), c=Cgt)
plt.title('2D Visualization of FACES with PCA') 
plt.show()


In [ ]:
# 3D Visualization
import plotly.graph_objects as go
data = go.Scatter3d(x=Xvis, y=Yvis, z=Zvis, mode='markers', marker=dict(size=10, color=C, colorscale='jet', opacity=1)) # data as points
# data = go.Scatter3d(x=Xvis, y=Yvis, z=Zvis, mode='markers', marker=dict(size=1, color=C, colorscale='jet', opacity=1, showscale=True)) # w/ colobar 
fig = go.Figure(data=[data]) 
fig.update_layout(margin=dict(l=0, r=0, b=0, t=30, pad=0)) # tight layout but t=25 required for showing title 
fig.update_layout(autosize=False, width=800, height=800, title_text="PCA FACES") # figure size and title
# fig.update_layout(scene = dict(xaxis = dict(visible=False), yaxis = dict(visible=False), zaxis = dict(visible=False))) # no grid, no axis 
# fig.update_layout(scene = dict(xaxis_title = ' ', yaxis_title = ' ', zaxis_title = ' ')) # no axis name 
fig.update_layout(scene = dict(zaxis = dict(showgrid = True, showticklabels = False), zaxis_title = ' ') ) # no range values, no axis name, grid on
fig.update_layout(scene = dict(yaxis = dict(showgrid = True, showticklabels = False), yaxis_title = ' ') ) # no range values, no axis name, grid on
fig.update_layout(scene = dict(xaxis = dict(showgrid = True, showticklabels = False), xaxis_title = ' ') ) # no range values, no axis name, grid on
fig.show()


In [ ]:
# Generate new faces with the equation : x(\alpha) = mean_X + \alpha* PD(k)
#  where $mean_X$ is the mean of the dataset, and $PD(k)$ is the $k^{th}$ principal directions

# Generate new faces
meanX = np.mean(X,axis=0) 

plt.figure(14)
k = 0; new_face = meanX - 0.025* EnPD[k]* PD[k,:]
rotated_img = scipy.ndimage.rotate(np.reshape(new_face,[Nx,Ny]), -90)
plt.subplot(331).imshow(rotated_img, interpolation='nearest', cmap='Greys_r')
plt.title('Mean Face - alpha* PD(1)')
plt.axis('off')
k = 0; new_face = meanX - 0.0* EnPD[k]* PD[k,:]
rotated_img = scipy.ndimage.rotate(np.reshape(new_face,[Nx,Ny]), -90)
plt.subplot(332).imshow(rotated_img, interpolation='nearest', cmap='Greys_r')
plt.title('Mean Face')
plt.axis('off')
k = 0; new_face = meanX + 0.025* EnPD[k]* PD[k,:]
rotated_img = scipy.ndimage.rotate(np.reshape(new_face,[Nx,Ny]), -90)
plt.subplot(333).imshow(rotated_img, interpolation='nearest', cmap='Greys_r')
plt.title('Mean Face + alpha* PD(1)')
plt.axis('off')
k = 1; new_face = meanX - 0.025* EnPD[k]* PD[k,:]
rotated_img = scipy.ndimage.rotate(np.reshape(new_face,[Nx,Ny]), -90)
plt.subplot(334).imshow(rotated_img, interpolation='nearest', cmap='Greys_r')
plt.title('Mean Face - alpha* PD(2)')
plt.axis('off')
k = 1; new_face = meanX - 0.0* EnPD[k]* PD[k,:]
rotated_img = scipy.ndimage.rotate(np.reshape(new_face,[Nx,Ny]), -90)
plt.subplot(335).imshow(rotated_img, interpolation='nearest', cmap='Greys_r')
plt.title('Mean Face')
plt.axis('off')
k = 1; new_face = meanX + 0.025* EnPD[k]* PD[k,:]
rotated_img = scipy.ndimage.rotate(np.reshape(new_face,[Nx,Ny]), -90)
plt.subplot(336).imshow(rotated_img, interpolation='nearest', cmap='Greys_r')
plt.title('Mean Face + alpha* PD(2)')
plt.axis('off')
k = 2; new_face = meanX - 0.025* EnPD[k]* PD[k,:]
rotated_img = scipy.ndimage.rotate(np.reshape(new_face,[Nx,Ny]), -90)
plt.subplot(337).imshow(rotated_img, interpolation='nearest', cmap='Greys_r')
plt.title('Mean Face - alpha* PD(3)')
plt.axis('off')
k = 2; new_face = meanX - 0.0* EnPD[k]* PD[k,:]
rotated_img = scipy.ndimage.rotate(np.reshape(new_face,[Nx,Ny]), -90)
plt.subplot(338).imshow(rotated_img, interpolation='nearest', cmap='Greys_r')
plt.title('Mean Face')
plt.axis('off')
k = 2; new_face = meanX + 0.025* EnPD[k]* PD[k,:]
rotated_img = scipy.ndimage.rotate(np.reshape(new_face,[Nx,Ny]), -90)
plt.subplot(339).imshow(rotated_img, interpolation='nearest', cmap='Greys_r')
plt.title('Mean Face + alpha* PD(3)')
plt.axis('off')


# PCA for MNIST

In [ ]:
# MNIST dataset
mat = scipy.io.loadmat('datasets/MNIST_data.mat')
X = Xnumpy = mat['X']
n = X.shape[0]
d = X.shape[1]
C = mat['C'].squeeze()
print(n,d)

# PCA
nb_pca = 3
[PC,PD,EnPD] = compute_pca(X,nb_pca)
Xvis = PC[:,0]
Yvis = PC[:,1]
Zvis = PC[:,2]

# 2D Visualization
plt.figure(3)
plt.scatter(Xvis, Yvis, c=C, s=1, color=pyplot.jet())
plt.show()


In [ ]:
# 3D Visualization
import plotly.graph_objects as go
data = go.Scatter3d(x=Xvis, y=Yvis, z=Zvis, mode='markers', marker=dict(size=1, color=C, colorscale='jet', opacity=1)) # data as points
# data = go.Scatter3d(x=Xvis, y=Yvis, z=Zvis, mode='markers', marker=dict(size=1, color=C, colorscale='jet', opacity=1, showscale=True)) # w/ colobar 
fig = go.Figure(data=[data]) 
fig.update_layout(margin=dict(l=0, r=0, b=0, t=30, pad=0)) # tight layout but t=25 required for showing title 
fig.update_layout(autosize=False, width=800, height=800, title_text="PCA MNIST") # figure size and title
# fig.update_layout(scene = dict(xaxis = dict(visible=False), yaxis = dict(visible=False), zaxis = dict(visible=False))) # no grid, no axis 
# fig.update_layout(scene = dict(xaxis_title = ' ', yaxis_title = ' ', zaxis_title = ' ')) # no axis name 
fig.update_layout(scene = dict(zaxis = dict(showgrid = True, showticklabels = False), zaxis_title = ' ') ) # no range values, no axis name, grid on
fig.update_layout(scene = dict(yaxis = dict(showgrid = True, showticklabels = False), yaxis_title = ' ') ) # no range values, no axis name, grid on
fig.update_layout(scene = dict(xaxis = dict(showgrid = True, showticklabels = False), xaxis_title = ' ') ) # no range values, no axis name, grid on
fig.show()
